<a href="https://colab.research.google.com/github/atomicSteiner/HealthcareSBERT/blob/main/healthcareSBERTbase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import numpy as np

# Load OHSUMED
ds = load_dataset("community-datasets/ohsumed")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ohsumed/train-00000-of-00001.parquet:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

ohsumed/test-00000-of-00001.parquet:   0%|          | 0.00/181M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54709 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/293855 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['seq_id', 'medline_ui', 'mesh_terms', 'title', 'publication_type', 'abstract', 'author', 'source'],
        num_rows: 54709
    })
    test: Dataset({
        features: ['seq_id', 'medline_ui', 'mesh_terms', 'title', 'publication_type', 'abstract', 'author', 'source'],
        num_rows: 293855
    })
})


filtering the train part

In [ ]:
if('ohsumed_cleaned_train.csv' in os.listdir()):
  df_train = pd.read_csv("ohsumed_cleaned_train.csv")
else:
  df_train = pd.DataFrame(ds["train"])
  #print(df_train.head())
  #print(df_train.info())
  df_train = df_train[["seq_id", 'medline_ui', "mesh_terms", "title", "abstract"]]
  #'seq_id', 'medline_ui', 'mesh_terms', 'title', 'publication_type', 'abstract', 'author', 'source'
  print("Before removing empty abstracts:", len(df_train))
  # Drop rows with empty or null abstracts
  df_train = df_train.dropna(subset=["abstract"])
  df_train = df_train[df_train["abstract"].str.strip().astype(bool)]
  print("After removing empty abstracts:", len(df_train))
  before = len(df_train)
  df_train = df_train.drop_duplicates(subset=["abstract"])
  after = len(df_train)
  print(f"Removed {before - after} duplicate abstracts.")
  print("Remaining abstracts:", len(df_train))
  df_train.to_csv("ohsumed_cleaned_train.csv", index=False)

Before removing empty abstracts: 54709
After removing empty abstracts: 54709
Removed 17821 duplicate abstracts.
Remaining abstracts: 36888


now filtering test part

In [ ]:
if('ohsumed_cleaned_test.csv' in os.listdir()):
  df_test = pd.read_csv("ohsumed_cleaned_test.csv")
else:
  df_test = pd.DataFrame(ds["test"])
  print(df_test.head())
  print(df_test.info())
  df_test = df_test[["seq_id", 'medline_ui', "mesh_terms", "title", "abstract"]]
  print("Before removing empty abstracts:", len(df_test))
  before = len(df_test)
  df_test = df_test.drop_duplicates(subset=["abstract"])
  after = len(df_test)
  print(f"Removed {before - after} duplicate abstracts.")
  print("Remaining abstracts:", len(df_test))
  df_test.to_csv("ohsumed_cleaned_test.csv", index=False)

   seq_id  medline_ui                                         mesh_terms  \
0   54711    88000001  Acetaldehyde/*ME; Buffers; Catalysis; HEPES/PD...   
1   54711    88000002  Adult; Alcohol, Ethyl/*AN; Breath Tests/*; Hum...   
2   54711    88000003  Alcoholism/*PP; Animal; Diprenorphine/PD; Fema...   
3   54711    88000006  Adult; Alcohol Drinking/*PH; Alcoholism/*BL/CO...   
4   54711    88000007  Adult; Alcoholism/*BL; Blood Platelets/*ME; Er...   

                                               title  publication_type  \
0  The binding of acetaldehyde to the active site...  JOURNAL ARTICLE.   
1  Reductions in breath ethanol readings in norma...  JOURNAL ARTICLE.   
2  Does the blockade of opioid receptors influenc...  JOURNAL ARTICLE.   
3  Drinkwatchers--description of subjects and eva...  JOURNAL ARTICLE.   
4  Platelet affinity for serotonin is increased i...  JOURNAL ARTICLE.   

                                            abstract  \
0  Ribonuclease A was reacted with [1-13C,

#Create baseline embeddings

In [ ]:
#retreival via textual information
from sentence_transformers import SentenceTransformer, util
import torch

abstracts = df_test['abstract'].tolist()
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#Encode corpus into embeddings and save it
#or import it if it's already done

if('embeddings_baseline.npy' in os.listdir()):
  embeddings = np.load('embeddings_baseline.npy')
else:
  embeddings = model.encode(abstracts, batch_size=16, device='cuda', show_progress_bar=True, convert_to_numpy=True)
  np.save('embeddings_baseline.npy', embeddings)

Batches:   0%|          | 0/12282 [00:00<?, ?it/s]

#Fine-tune SBERT

In [ ]:
# Install sentence-transformers if not already
!pip install -q sentence-transformers

In [ ]:
# Imports
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import random
import ast

In [ ]:
i=1
print(df_train["mesh_terms"][i])
print()
df_train["mesh_terms"][i]

Step 1  Generate Positive Pairs (Improved MeSH Parsing). We are preparing the training data

In [ ]:
# -----------------------------
# 1.1 Improved MeSH parsing function
# -----------------------------
def parse_mesh_improved(mesh_str):
    if not isinstance(mesh_str, str):
        return []

    terms = mesh_str.split(';')  # split by semicolon
    clean_terms = []

    for t in terms:
        t = t.strip()            # remove leading/trailing whitespace
        if t == '':
            continue
        # keep only the part before '/' (ignore qualifiers like /*PO, /TH)
        t = t.split('/')[0].strip()
        clean_terms.append(t)

    return clean_terms

# Apply to dataframe
df_train['mesh_terms'] = df_train['mesh_terms'].apply(parse_mesh_improved)

# Check first few rows
print(df_train['mesh_terms'].head())


In [ ]:
# -----------------------------
# 1.2 Group abstracts by MeSH term
# -----------------------------
mesh_to_abstracts = {}
for _, row in df_train.iterrows():
    abstract = row['abstract']
    for mesh in row['mesh_terms']:
        if mesh not in mesh_to_abstracts:
            mesh_to_abstracts[mesh] = []
        mesh_to_abstracts[mesh].append(abstract)

In [ ]:
# -----------------------------
# 1.3 Create positive pairs
# -----------------------------
positive_pairs = []
max_pairs_per_mesh = 50  # limit to avoid too many pairs

for abstracts in mesh_to_abstracts.values():
    if len(abstracts) < 2:
        continue
    # sample abstracts if group is very large
    abstracts_sample = random.sample(abstracts, min(len(abstracts), max_pairs_per_mesh))
    # create all pairwise combinations
    for i in range(len(abstracts_sample)):
        for j in range(i+1, len(abstracts_sample)):
            positive_pairs.append((abstracts_sample[i], abstracts_sample[j]))

print(f"Generated {len(positive_pairs)} positive pairs.")

In [ ]:
# -----------------------------
# 1.4 Convert to SBERT InputExample objects
# -----------------------------
train_examples = [InputExample(texts=[a1, a2]) for a1, a2 in positive_pairs]

In [ ]:
# Quick check
print(train_examples[:5])

Step 2 and others...

In [ ]:
# -----------------------------
# 2. Create DataLoader
# -----------------------------
batch_size = 16
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

In [ ]:
# -----------------------------
# 3. Load pre-trained SBERT
# -----------------------------
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)
model.to('cuda')  # use GPU
#TODO add a check if the model is already loaded

In [ ]:
# -----------------------------
# 4. Define contrastive loss
# -----------------------------
train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
# -----------------------------
# 5. Fine-tune SBERT
# -----------------------------
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path='fine_tuned_sbert_ohsumed'
)

In [ ]:
# -----------------------------
# 6. Save the fine-tuned model
# -----------------------------
# Already saved by `output_path` above
print("Fine-tuned SBERT saved at 'fine_tuned_sbert_ohsumed'")

----------------------------------------------------------------------------------------------------------------

i had to change the sizes because it took too much memory

In [ ]:
os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
# -----------------------------
# 1️⃣ Improved MeSH parsing
# -----------------------------
def parse_mesh_improved(mesh_str):
    if not isinstance(mesh_str, str):
        return []

    terms = mesh_str.split(';')  # split by semicolon
    clean_terms = []

    for t in terms:
        t = t.strip()            # remove whitespace
        if t == '':
            continue
        # keep only main term (before '/')
        t = t.split('/')[0].strip()
        clean_terms.append(t)

    return clean_terms

df_train['mesh_terms'] = df_train['mesh_terms'].apply(parse_mesh_improved)


In [ ]:
# -----------------------------
# 2️⃣ Generate positive pairs
# -----------------------------
mesh_to_abstracts = {}
for _, row in df_train.iterrows():
    abstract = row['abstract']
    for mesh in row['mesh_terms']:
        if mesh not in mesh_to_abstracts:
            mesh_to_abstracts[mesh] = []
        mesh_to_abstracts[mesh].append(abstract)

positive_pairs = []
max_pairs_per_mesh = 20  # lower than before to reduce memory

for abstracts in mesh_to_abstracts.values():
    if len(abstracts) < 2:
        continue
    abstracts_sample = random.sample(abstracts, min(len(abstracts), max_pairs_per_mesh))
    for i in range(len(abstracts_sample)):
        for j in range(i+1, len(abstracts_sample)):
            positive_pairs.append((abstracts_sample[i], abstracts_sample[j]))

print(f"Generated {len(positive_pairs)} positive pairs.")

train_examples = [InputExample(texts=[a1, a2]) for a1, a2 in positive_pairs]

# Optional: sample a subset if still too large
if len(train_examples) > 20000:
    train_examples = random.sample(train_examples, 20000)
    print(f"Subsampled to {len(train_examples)} training examples.")


In [ ]:
# -----------------------------
# 3️⃣ DataLoader
# -----------------------------
batch_size = 8  # small to reduce GPU memory
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)


In [ ]:
# -----------------------------
# 4️⃣ Load pre-trained SBERT
# -----------------------------
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)
model.to('cuda')

In [ ]:
# -----------------------------
# 5️⃣ Define contrastive loss
# -----------------------------
train_loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
# -----------------------------
# 6️⃣ Fine-tune SBERT
# -----------------------------
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path='fine_tuned_sbert_ohsumed',
    optimizer_params={'lr':2e-5},
    use_amp=True  # mixed-precision for memory efficiency
)

In [ ]:
# -----------------------------
# 7️⃣ Save fine-tuned model
# -----------------------------
# Already saved at output_path
print("Fine-tuned SBERT saved at 'fine_tuned_sbert_ohsumed'")

In [ ]:
!zip -r fine_tuned_sbert_ohsumed.zip fine_tuned_sbert_ohsumed
from google.colab import files
files.download('fine_tuned_sbert_ohsumed.zip')


more things

In [ ]:
"""#Compute cosine similarities
cosine_scores = util.cos_sim(query_embedding, embeddings)
#

#Retrieve top-k most similar documents
top_k = min(3, len(embeddings))  # top 3 matches
top_results = torch.topk(cosine_scores, k=top_k)

#Print results
print(f"Query: {query}\n")
for score, idx in zip(top_results.values[0], top_results.indices[0]):
    print(f"{embeddings[idx]} (Score: {score:.4f})")"""

In [ ]:
"""#Define a query
query = "blood"
query_embedding = model.encode(query, convert_to_tensor=True)